In [1]:
import json

In [2]:
metrics_path = "D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_metrics.json"

with open(metrics_path, "r") as f:
    metrics = json.load(f)

In [3]:
def get_value(metric_key, year):
    return metrics[metric_key]["values"][str(year)]


def percentage_change(current, previous):
    return ((current - previous) / previous) * 100


def margin(part, total):
    return (part / total) * 100


def safe_round(value):
    return round(value, 2)


def format_money(value):
    return f"${value:,.0f} million"


def format_percent(value):
    return f"{value:.2f}%"

In [4]:
def generate_financial_analysis(year=2023):
    
    previous_year = year - 1

    # Revenue
    revenue = get_value("total_net_sales", year)
    prev_revenue = get_value("total_net_sales", previous_year)

    revenue_growth = percentage_change(revenue, prev_revenue)

    # Net income
    net_income = get_value("net_income", year)
    prev_net_income = get_value("net_income", previous_year)

    net_income_growth = percentage_change(
        net_income,
        prev_net_income
    )

    # Margins
    gross_margin_value = get_value("gross_margin", year)
    operating_income = get_value("operating_income", year)

    gross_margin_pct = margin(
        gross_margin_value,
        revenue
    )

    operating_margin_pct = margin(
        operating_income,
        revenue
    )

    net_profit_margin_pct = margin(
        net_income,
        revenue
    )

    # Expenses
    rd = get_value("research_and_development", year)

    sga = get_value(
        "selling_general_and_administrative",
        year
    )

    total_opex = get_value(
        "total_operating_expenses",
        year
    )

    rd_pct = margin(rd, revenue)

    sga_pct = margin(sga, revenue)

    opex_pct = margin(total_opex, revenue)

    # Balance sheet
    assets = get_value("total_assets", year)

    liabilities = get_value(
        "total_liabilities",
        year
    )

    cash = get_value(
        "cash_and_cash_equivalents",
        year
    )

    liabilities_to_assets = margin(
        liabilities,
        assets
    )

    cash_to_assets = margin(
        cash,
        assets
    )

    analysis = {
        "year": year,

        "revenue": {
            "value": revenue,
            "growth_percent": safe_round(revenue_growth)
        },

        "net_income": {
            "value": net_income,
            "growth_percent": safe_round(net_income_growth)
        },

        "profitability": {
            "gross_margin_percent": safe_round(gross_margin_pct),
            "operating_margin_percent": safe_round(operating_margin_pct),
            "net_profit_margin_percent": safe_round(net_profit_margin_pct)
        },

        "expense_efficiency": {
            "rd_as_percent_of_sales": safe_round(rd_pct),
            "sga_as_percent_of_sales": safe_round(sga_pct),
            "operating_expenses_as_percent_of_sales": safe_round(opex_pct)
        },

        "balance_sheet": {
            "liabilities_to_assets_percent": safe_round(liabilities_to_assets),
            "cash_to_assets_percent": safe_round(cash_to_assets)
        }
    }

    return analysis

In [5]:
analysis_2023 = generate_financial_analysis(2023)

analysis_2023

{'year': 2023,
 'revenue': {'value': 383285.0, 'growth_percent': -2.8},
 'net_income': {'value': 96995.0, 'growth_percent': -2.81},
 'profitability': {'gross_margin_percent': 44.13,
  'operating_margin_percent': 3.15,
  'net_profit_margin_percent': 25.31},
 'expense_efficiency': {'rd_as_percent_of_sales': 7.8,
  'sga_as_percent_of_sales': 6.5,
  'operating_expenses_as_percent_of_sales': 14.31},
 'balance_sheet': {'liabilities_to_assets_percent': 82.37,
  'cash_to_assets_percent': 8.5}}

In [6]:
def build_financial_summary(analysis):

    year = analysis["year"]

    revenue = analysis["revenue"]["value"]
    revenue_growth = analysis["revenue"]["growth_percent"]

    net_income = analysis["net_income"]["value"]
    net_income_growth = analysis["net_income"]["growth_percent"]

    gross_margin = analysis["profitability"]["gross_margin_percent"]

    operating_margin = analysis["profitability"]["operating_margin_percent"]

    net_margin = analysis["profitability"]["net_profit_margin_percent"]

    rd_pct = analysis["expense_efficiency"]["rd_as_percent_of_sales"]

    liabilities_ratio = analysis["balance_sheet"]["liabilities_to_assets_percent"]

    cash_ratio = analysis["balance_sheet"]["cash_to_assets_percent"]

    summary = f"""
APPLE FINANCIAL ANALYSIS ({year})

Revenue:
- Revenue: {format_money(revenue)}
- Revenue growth: {format_percent(revenue_growth)}

Net Income:
- Net income: {format_money(net_income)}
- Net income growth: {format_percent(net_income_growth)}

Profitability:
- Gross margin: {format_percent(gross_margin)}
- Operating margin: {format_percent(operating_margin)}
- Net profit margin: {format_percent(net_margin)}

Expense Efficiency:
- R&D as % of sales: {format_percent(rd_pct)}

Balance Sheet:
- Liabilities to assets: {format_percent(liabilities_ratio)}
- Cash to assets: {format_percent(cash_ratio)}
"""

    return summary

In [7]:
summary = build_financial_summary(
    analysis_2023
)

print(summary)


APPLE FINANCIAL ANALYSIS (2023)

Revenue:
- Revenue: $383,285 million
- Revenue growth: -2.80%

Net Income:
- Net income: $96,995 million
- Net income growth: -2.81%

Profitability:
- Gross margin: 44.13%
- Operating margin: 3.15%
- Net profit margin: 25.31%

Expense Efficiency:
- R&D as % of sales: 7.80%

Balance Sheet:
- Liabilities to assets: 82.37%
- Cash to assets: 8.50%



In [8]:
output_path = "D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_analysis.json"

with open(output_path, "w") as f:
    json.dump(analysis_2023, f, indent=4)

output_path

'D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_analysis.json'

In [9]:
import ollama

In [16]:
def generate_llm_financial_interpretation(summary):

    prompt = f"""
You are a senior equity research analyst.

Interpret the following Apple financial analysis.

Rules:
1. Do not make claims not directly supported by the metrics.
2. Do not compare with industry averages unless benchmark data is provided.
3. Do not say liabilities exceed assets unless ratio > 100%.
4. Use cautious analyst-style language.
5. Separate observations from conclusions.

Focus on:
- profitability
- growth
- efficiency
- financial health

Keep the answer concise but insightful.
Do not compare with industry average unless benchmark data is provided.

Financial summary:
{summary}
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    return response["message"]["content"]

In [17]:
interpretation = generate_llm_financial_interpretation(
    summary
)

print(interpretation)

 In the provided financial analysis for Apple in 2023, several key observations can be made regarding profitability, growth, efficiency, and financial health:

1. Profitability: The company demonstrates a strong gross margin of 44.13%, indicating a high level of pricing power or cost control over its products. However, the operating margin (3.15%) and net profit margin (25.31%) show a decrease compared to previous years, suggesting potential operational inefficiencies or increased expenses.

2. Growth: Both revenue and net income have experienced a decline of -2.80% and -2.81%, respectively, indicating a contraction in the business over the year. This negative growth may be a cause for concern for investors.

3. Efficiency: Apple invests 7.80% of its sales into research and development (R&D), which is within the typical range for technology companies. This investment in innovation could help drive future growth, but it also represents a significant expense.

4. Financial Health: The li

In [18]:
print(get_value("operating_income", 2023))

print(get_value("total_net_sales", 2023))

12066.0
383285.0


In [19]:
print(
    margin(
        get_value("operating_income", 2023),
        get_value("total_net_sales", 2023)
    )
)

3.1480491018432764


In [20]:
metrics["operating_income"]

{'label': 'operating income',
 'values': {'2023': 12066.0, '2022': 11569.0, '2021': 9817.0},
 'source_page': 50,
 'source_type': 'pymupdf_financial_line',
 'raw_text': ''}

In [21]:
df[
    df["metric_name"].str.contains(
        "operating income",
        case=False,
        na=False
    )
]

NameError: name 'df' is not defined

In [22]:
print(locals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'json', '_i2', 'metrics_path', 'f', 'metrics', '_i3', 'get_value', 'percentage_change', 'margin', 'safe_round', 'format_money', 'format_percent', '_i4', 'generate_financial_analysis', '_i5', 'analysis_2023', '_5', '_i6', 'build_financial_summary', '_i7', 'summary', '_i8', 'output_path', '_8', '_i9', 'ollama', '_i10', 'generate_llm_financial_interpretation', '_i11', '_i12', '_i13', 'interpretation', '_i14', '_i15', '_i16', '_i17', '_i18', '_i19', '_i20', '_20', '_i21', '_i22'])


In [28]:
import pandas as pd

csv_path = "D:/sudhendra/learning projects/GraphRAG/data/apple_2023_pymupdf_financial_lines.csv"

df = pd.read_csv(csv_path)

df.head()

,page,metric_name,2023,2022,2021,raw_label,raw_values
0,3,principal accountant fees and services,53.0,54.0,57.0,Principal Accountant Fees and Services,"[53.0, 54.0, 57.0]"
1,24,total net sales,383285.0,394328.0,365817.0,Total net sales,"[383285.0, 394328.0, 365817.0]"
2,25,iphone (1),200583.0,205489.0,191973.0,iPhone (1),"[200583.0, 205489.0, 191973.0]"
3,25,mac (1),29357.0,40177.0,35190.0,Mac (1),"[29357.0, 40177.0, 35190.0]"
4,25,ipad (1),28300.0,29292.0,31862.0,iPad (1),"[28300.0, 29292.0, 31862.0]"


In [29]:
df[
    df["metric_name"].str.contains(
        "operating income",
        case=False,
        na=False
    )
]

,page,metric_name,2023,2022,2021,raw_label,raw_values
28,31,operating income,114301.0,119437.0,108949.0,Operating income,"[114301.0, 119437.0, 108949.0]"
53,50,operating income,60508.0,62683.0,53382.0,Operating income,"[60508.0, 62683.0, 53382.0]"
54,50,operating income,36098.0,35233.0,32505.0,Operating income,"[36098.0, 35233.0, 32505.0]"
55,50,operating income,30328.0,31153.0,28504.0,Operating income,"[30328.0, 31153.0, 28504.0]"
56,50,operating income,11888.0,12257.0,12798.0,Operating income,"[11888.0, 12257.0, 12798.0]"
57,50,operating income,12066.0,11569.0,9817.0,Operating income,"[12066.0, 11569.0, 9817.0]"
58,50,segment operating income,150888.0,152895.0,137006.0,Segment operating income,"[150888.0, 152895.0, 137006.0]"
60,50,total operating income,114301.0,119437.0,108949.0,Total operating income,"[114301.0, 119437.0, 108949.0]"


In [30]:
matched_rows = df[
    df["metric_name_clean"] == "operating income"
]

matched_row = matched_rows.iloc[0]

KeyError: 'metric_name_clean'

In [32]:
df["metric_name_clean"] = (
    df["metric_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [33]:
matched_rows = df[
    df["metric_name_clean"] == "operating income"
]

matched_row = matched_rows.iloc[0]

matched_row

page                                             31
metric_name                        operating income
2023                                       114301.0
2022                                       119437.0
2021                                       108949.0
raw_label                          Operating income
raw_values           [114301.0, 119437.0, 108949.0]
metric_name_clean                  operating income
Name: 28, dtype: object

In [35]:
metrics["operating_income"] = {
    "label": "operating income",
    "values": {
        "2023": float(matched_row["2023"]),
        "2022": float(matched_row["2022"]),
        "2021": float(matched_row["2021"])
    },
    "source_page": int(matched_row["page"]),
    "source_type": "pymupdf_financial_line",
    "raw_text": matched_row.get("raw_label", "")
}

In [36]:
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

In [37]:
with open(metrics_path, "r") as f:
    metrics = json.load(f)

In [38]:
print(get_value("operating_income", 2023))

114301.0


In [39]:
analysis_2023 = generate_financial_analysis(2023)
summary = build_financial_summary(analysis_2023)

print(summary)


APPLE FINANCIAL ANALYSIS (2023)

Revenue:
- Revenue: $383,285 million
- Revenue growth: -2.80%

Net Income:
- Net income: $96,995 million
- Net income growth: -2.81%

Profitability:
- Gross margin: 44.13%
- Operating margin: 29.82%
- Net profit margin: 25.31%

Expense Efficiency:
- R&D as % of sales: 7.80%

Balance Sheet:
- Liabilities to assets: 82.37%
- Cash to assets: 8.50%



In [42]:
def generate_llm_financial_interpretation(summary):

    prompt = f"""
You are a senior equity research analyst.

Interpret the following Apple financial analysis.

def interpret_liabilities_ratio(ratio):

    if ratio < 50:
        return "The company maintains relatively low liabilities compared to assets."

    elif ratio < 90:
        return "The company carries substantial liabilities relative to assets, but liabilities remain below total assets."

    else:
        return "The company carries very high liabilities relative to assets."
        
Rules:
1. Do not make claims not directly supported by the metrics.
2. Do not compare with industry averages unless benchmark data is provided.
3. Do not say liabilities exceed assets unless ratio > 100%.
4. Use cautious analyst-style language.
5. Separate observations from conclusions.

Focus on:
- profitability
- growth
- efficiency
- financial health

Keep the answer concise but insightful.
Do not compare with industry average unless benchmark data is provided.

Financial summary:
{summary}
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    return response["message"]["content"]

In [43]:
interpretation = generate_llm_financial_interpretation(
    summary
)

print(interpretation)

 In the provided Apple financial analysis for 2023, we can observe several key points regarding profitability, growth, efficiency, and financial health.

Profitability:
- The gross margin of 44.13%, operating margin of 29.82%, and net profit margin of 25.31% indicate a strong profitability performance, with the company generating substantial profits from its operations.

Growth:
- A decline in both revenue (-2.80%) and net income (-2.81%) suggests a contraction in Apple's top and bottom lines compared to the previous year.

Efficiency:
- Research and development (R&D) expenses account for 7.80% of sales, indicating that Apple invests significantly in innovation and product development.

Financial Health:
- The liabilities to assets ratio of 82.37% indicates that the company carries substantial liabilities relative to its assets, suggesting a relatively high leverage position. However, it's important to note that this ratio is below 100%, meaning liabilities do not exceed total assets.
